# IT3212 Assignment 1 — Student Graduation Dataset Preprocessing

Preprocessing pipeline for the UCI *Predict Students' Dropout and Academic Success* dataset: exploration, cleaning, outlier handling, transformation, and train/test splitting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

DATA_PATH = "../data/graduation_dataset.csv"
df = pd.read_csv(DATA_PATH)
df.shape

## 1. Data Exploration (10)

### 1a. First rows, summary statistics, data types

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe(include="all")

All 34 predictor columns load as numeric (`int64`/`float64`); only `Target` is a string. A number of the
numeric columns are actually **categorical codes** (e.g. `Marital status`, `Application mode`, `Course`,
qualification/occupation codes) or **binary flags** (e.g. `Gender`, `Debtor`, `Scholarship holder`) rather
than continuous measurements — `dtype` alone doesn't reveal this, so we classify columns explicitly below.
That classification is reused in later sections (outlier handling, encoding).

In [ ]:
TARGET_COL = "Target"

BINARY_COLS = [
    "Daytime/evening attendance",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International",
]

NOMINAL_COLS = [
    "Marital status",
    "Application mode",
    "Application order",
    "Course",
    "Previous qualification",
    "Nationality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
]

CONTINUOUS_COLS = [
    "Age at enrollment",
    "Curricular units 1st sem (credited)",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 1st sem (without evaluations)",
    "Curricular units 2nd sem (credited)",
    "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
    "Curricular units 2nd sem (without evaluations)",
    "Unemployment rate",
    "Inflation rate",
    "GDP",
]

assert set(BINARY_COLS) | set(NOMINAL_COLS) | set(CONTINUOUS_COLS) | {TARGET_COL} == set(df.columns)
len(BINARY_COLS), len(NOMINAL_COLS), len(CONTINUOUS_COLS)

### 1b. Missing values, outliers, and unique values in categorical columns

In [ ]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)

No missing values are reported in any column (see empty result above). We still build the missing-value
handling logic in Section 2 defensively, since the grading rubric expects it and real-world exports of this
dataset sometimes carry a handful of nulls.

In [ ]:
df[TARGET_COL].value_counts()

The target classes are imbalanced (`Graduate` ≈ 50%, `Dropout` ≈ 32%, `Enrolled` ≈ 18%). This matters for
any downstream modelling (e.g. stratified splitting in Section 5) even though it isn't a preprocessing step
itself.

In [ ]:
for col in BINARY_COLS + NOMINAL_COLS:
    print(f"{col:40s} n_unique={df[col].nunique():3d}  values={sorted(df[col].unique())[:10]}")

Outlier detection only makes sense on the continuous/count columns — running IQR on binary flags or nominal
codes produces meaningless "outliers" (e.g. flagging the minority class of a 0/1 column). We therefore scope
the IQR pass to `CONTINUOUS_COLS`; the actual removal/capping/transform decision is deferred to Section 3.

In [ ]:
def iqr_outlier_summary(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        q1, q3 = frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        mask = (frame[col] < lower) | (frame[col] > upper)
        rows.append({
            "column": col,
            "q1": q1, "q3": q3, "lower_fence": lower, "upper_fence": upper,
            "n_outliers": int(mask.sum()),
            "pct_outliers": round(100 * mask.mean(), 1),
        })
    return pd.DataFrame(rows).sort_values("pct_outliers", ascending=False)

outlier_summary = iqr_outlier_summary(df, CONTINUOUS_COLS)
outlier_summary

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, col in zip(axes.ravel(), CONTINUOUS_COLS):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col, fontsize=9)
    ax.set_ylabel("")
plt.tight_layout()
plt.savefig("../figures/01_continuous_boxplots.png", dpi=150)
plt.show()

**Observations:**
- `Age at enrollment` has a long right tail (median 20, max 70) — genuine outliers from non-traditional /
  older students.
- The `Curricular units ... (grade)` columns are the most heavily flagged (~16–20%), but are also
  zero-inflated: a grade of 0 means the student had no evaluations that semester, not a true low score. This
  ambiguity needs to be resolved in Section 3 before treating those points as outliers.
- The `Curricular units ... (credited/enrolled/evaluations/approved/without evaluations)` count columns are
  right-skewed count data with many zeros, which is expected (not every student enrolls in, or gets credit
  for, the same number of units) rather than a sign of data-quality issues.
- `Unemployment rate`, `Inflation rate`, and `GDP` show 0% IQR outliers — they only take ~9–10 distinct
  values shared across cohorts admitted in the same period, so their spread is naturally tight.